# Implementation of a PRNG-Based Stream Cipher with Dynamic Seed Derivation

Author: Drei Abmab

### Brief Explanation

This code demonstrates a custom implementation of a stream cipher using a pseudorandom number generator (PRNG). It shows how to generate a keystream, encrypt plaintext using XOR, decrypt ciphertext, and securely derive a seed using hashing and a nonce.

### Objective (Bullet Points)
- Simulate how stream ciphers operate
- Generate a deterministic keystream from an initial seed
- Use XOR operation for both encryption and decryption
- Ensure reversibility of encryption through XOR properties
- Enhance security by deriving the seed dynamically
- Combine a nonce and secret to produce a unique seed per session
- Demonstrate basic principles of PRNG-based cryptography

Sources for Code: https://drive.google.com/file/d/1UmBmKrm7ax-xWpIIlP-cmKEkDqA5SqnI/view?pli=1

## Library Imports

In [954]:
import os
import hashlib
import binascii
import random


This section imports the required Python libraries: os for system-level operations, hashlib for cryptographic hashing (used later for seed derivation), and binascii for converting between binary and hexadecimal representations.

--- 
## Section 1 : Custom PRNG (crand) Implementation

This section defines the crand(seed) function, which acts as a custom pseudorandom number generator. It initializes a list with the given seed and generates a sequence of numbers using a linear congruential generator (LCG) followed by additional mixing steps inspired by lagged Fibonacci generators.

In [955]:
def crand(seed):
    r=[]
    r.append(seed)
    for i in range(30):
        r.append((16807 * r[-1]) % 2147483647)
        if r[-1] < 0:
            r[-1] += 2147483647
    for i in range(31, 34):
        r.append(r[len(r)-31])
    for i in range(34, 344):
        r.append((r[len(r)-31] + r[len(r)-3]) % 2**32)
    while True:
        next = r[len(r)-31]+r[len(r)-3] % 2**32
        r.append(next)
        yield (next >> 1 if next < 2**32 else (next % 2**32) >> 1)

mygen = crand(2026)
firstfour = [next(mygen) for i in range(4)]
    
print(firstfour)

[1199659537, 1872465372, 1381520923, 36027969]


The generator continuously produces pseudorandom values using previous elements in the sequence and yields values after applying bit shifting to maintain consistency. This function is deterministic, meaning the same seed will always produce the same sequence, which is essential for both encryption and decryption.

--- 
## Section 2 : Encryption Using Keystream and XOR

This section demonstrates encryption using the generated keystream. A plaintext message is first converted into hexadecimal format, while multiple outputs from the PRNG are combined to form a hexadecimal key stream. The plaintext (as an integer) is then XORed with the key stream (also as an integer) to produce the ciphertext.

In [956]:
mygen = crand(2026)
rands = [next(mygen) for i in range(6)]
plaintext = b"Hello Cryptography"
hexplain = binascii.hexlify(plaintext)
hexkey = "".join(map(lambda x: format(x, 'x')[-6:], rands)) 
cipher_as_int = int(hexplain, 16) ^ int(hexkey, 16)
cipher_as_hex = format(cipher_as_int, 'x')

print("First Six(6):", rands)
print("Cipher as int:", cipher_as_int)
print("Cipher as hex:", cipher_as_hex)

First Six(6): [1199659537, 1872465372, 1381520923, 36027969, 1759202686, 784491491]
Cipher as int: 17531174702000424339215974891877867004758938
Cipher as hex: c93f7df7e2fc1b246255ca2ebc3f1fb20b9a


The result is displayed both as an integer and in hexadecimal form, illustrating how stream ciphers transform readable data into an unintelligible format.

Let's do it using A Random Number as seed:

In [957]:
seed  = random.randint(1000, 1000000)
mygen = crand(seed)
rands = [next(mygen) for i in range(6)]
plaintext = b"Hello Cryptography"
hexplain = binascii.hexlify(plaintext)
hexkey = "".join(map(lambda x: format(x, 'x')[-6:], rands)) 
cipher_as_int = int(hexplain, 16) ^ int(hexkey, 16)
cipher_as_hex = format(cipher_as_int, 'x')

print("seed", seed)
print("First Six(6):", rands)
print("Cipher as int:", cipher_as_int)
print("Cipher as hex:", cipher_as_hex)

seed 172908
First Six(6): [424713602, 1232469610, 478416543, 1574394141, 62977621, 1762016723]
Cipher as int: 2175401378375930946874864334567978147665322
Cipher as hex: 18f8ee19914ac77ce6a72d72a784347655aa


--- 
## Section 3 : Decryption Using XOR Reversal

This section shows how decryption works using the same PRNG and seed. The ciphertext is converted back into an integer and XORed again with the same keystream to recover the original plaintext. Since XOR is reversible, applying the same operation with the same key restores the original message.

In [958]:
cipher_hex = "c93f7df7e2fc1b246255ca2ebc3f1fb20b9a"

# Generate keystream using derived seed
num_rands = len(cipher_hex) // 6
mygen = crand(2026)
rands = [next(mygen) for _ in range(num_rands)]
hexkey = "".join(map(lambda x: format(x, 'x')[-6:], rands))

plain_as_int = int(cipher_hex, 16) ^ int(hexkey, 16) # XOR ciphertext with key to get plaintext

plain_as_hex = format(plain_as_int, 'x')
if len(plain_as_hex) % 2 != 0:
    plain_as_hex = '0' + plain_as_hex

plaintext = binascii.unhexlify(plain_as_hex).decode('utf-8')
print("Plaintext:", plaintext)

Plaintext: Hello Cryptography


 The result is converted back to a readable UTF-8 string, demonstrating the correctness of the encryption-decryption process.

Let's test the output with a different seeds.

In [959]:
cipher_hex = "c93f7df7e2fc1b246255ca2ebc3f1fb20b9a"

# Generate keystream using derived seed
num_rands = len(cipher_hex) // 6
list_of_keys = [1990, 2017, 2028, 2026, 9090, 1550, 3450, 4560, 2067, 4545]

for i in range(len(list_of_keys)):
    mygen = crand(list_of_keys[i])

    rands = [next(mygen) for _ in range(num_rands)]
    hexkey = "".join(map(lambda x: format(x, 'x')[-6:], rands))

    plain_as_int = int(cipher_hex, 16) ^ int(hexkey, 16) # XOR ciphertext with key to get plaintext

    plain_as_hex = format(plain_as_int, 'x')
    if len(plain_as_hex) % 2 != 0:
        plain_as_hex = '0' + plain_as_hex

    plaintext = binascii.unhexlify(plain_as_hex).decode('utf-8', errors='ignore') # Had to add ignore to avoid error
    print("Possible Plaintext:", plaintext)

Possible Plaintext: u8%ҠDvn`p
Possible Plaintext: /-ÂxCW;&6
Possible Plaintext: ]-gA/6de
Possible Plaintext: Hello Cryptography
Possible Plaintext: I-pwl aDp
Possible Plaintext: b^G>vn
Possible Plaintext: ت'dحHE
Possible Plaintext: Z`vOqŬ>K"v
Possible Plaintext: )E^ cWrsdA
Possible Plaintext: ek)}.W*Uޠ


In [ ]:
cipher_hex = "c93f7df7e2fc1b246255ca2ebc3f1fb20b9a"

# Generate keystream using derived seed
num_rands = len(cipher_hex) // 6

for i in range(10):
    seed = random.randint(1000, 9999)
    mygen = crand(seed)

    rands = [next(mygen) for _ in range(num_rands)]
    hexkey = "".join(map(lambda x: format(x, 'x')[-6:], rands))

    plain_as_int = int(cipher_hex, 16) ^ int(hexkey, 16) # XOR ciphertext with key to get plaintext

    plain_as_hex = format(plain_as_int, 'x')
    if len(plain_as_hex) % 2 != 0:
        plain_as_hex = '0' + plain_as_hex

    plaintext = binascii.unhexlify(plain_as_hex).decode('utf-8', errors='ignore') # Had to add ignore to avoid error
    print("Possible Plaintext:", plaintext, seed)

Possible Plaintext: ͪ/턬Ye 3006
Possible Plaintext: FySo{zf7)ӑ 3636
Possible Plaintext: `înI<
藙e; 3472
Possible Plaintext: 5+Y0	<|<_J 1634
Jossible Plaintext: 
ġknDe 3697
Possible Plaintext: _/v]KD?
, 2915
_[Gl 3453aintext: ,:
Possible Plaintext: +&2R.4J 3688
Possible Plaintext: hR1fDѮO_ 2397
Qa 1064 Plaintext: cK僅*(


--- 
## Section 4 : Seed Derivation with Nonce and Hashing

This section enhances security by deriving a new seed instead of using a fixed one. A nonce (a unique value used once) is combined with a secret value, converted into a properly formatted hexadecimal string, and then hashed using SHA-256. 

In [961]:
nonce = b"cc4304c09aee"
hexnonce = binascii.hexlify(nonce)
oursecret = 54321

concatenated_hex = hexnonce + format(oursecret, 'x').encode()
even_length = concatenated_hex.rjust(len(concatenated_hex) + len(concatenated_hex) % 2, b'0')

hexhash = hashlib.sha256(binascii.unhexlify(even_length)).hexdigest()
newseed = (int(hexhash, 16)) % 2**32
print(newseed)


3336748862


The resulting hash is reduced to a 32-bit integer to produce a new seed. This ensures that even if the same secret is reused, different nonces will generate different seeds, improving security and preventing keystream reuse.

--- 
## Section 5 : Secure Decryption with Derived Seed

This section demonstrates a more realistic decryption scenario where the nonce is embedded within the transmitted data. The nonce is extracted from the beginning of the message, and the same seed derivation process is applied using the shared secret. With the derived seed, the PRNG regenerates the identical keystream used during encryption. 

In [962]:
import binascii, hashlib


full_hex = "3e08816f1377f89f1c596fc197dd52946c92577bfd7c25c3"

# Split nonce (first 12 hex chars = 6 bytes) from ciphertext
hexnonce = full_hex[:12].encode()
cipher_hex = full_hex[12:]

# Derive new seed using same method
oursecret = 61983
concatenated_hex = hexnonce + format(oursecret, 'x').encode()
even_length = concatenated_hex.rjust(len(concatenated_hex) + len(concatenated_hex) % 2, b'0')
hexhash = hashlib.sha256(binascii.unhexlify(even_length)).hexdigest()
newseed = int(hexhash, 16) % 2**32
print("Derived seed:", newseed)

# Generate keystream using derived seed
num_rands = len(cipher_hex) // 6
mygen = crand(newseed)
rands = [next(mygen) for _ in range(num_rands)]
hexkey = "".join(map(lambda x: format(x, 'x')[-6:], rands))

# XOR to decrypt
plain_as_int = int(cipher_hex, 16) ^ int(hexkey, 16)
plain_as_hex = format(plain_as_int, 'x')
if len(plain_as_hex) % 2 != 0:
    plain_as_hex = '0' + plain_as_hex

plaintext = binascii.unhexlify(plain_as_hex).decode('utf-8')
print("Plaintext:", plaintext)

Derived seed: 42847799
Plaintext: this is a message.


The ciphertext is then XORed with this keystream to recover the original plaintext message, showing how both parties can securely communicate as long as they share the secret.

--- 
# Conclusion

Overall, this code illustrates the fundamental principles of stream ciphers: deterministic keystream generation, XOR-based encryption/decryption, and the importance of seed management. By incorporating nonce-based seed derivation, the implementation avoids keystream reuse, making the system more secure and closer to real-world cryptographic practices.